# 🧠 Multimodal Price Prediction Model
### Combines ViT Image Features + GRU Text Features → Predict Price (EGP) & Label

**Architecture:**
```
[ViT Image Features  (768-dim)] ──┐
                                   ├──► Concat (832) ──► FC(512)─ReLU ──► FC(256)─ReLU ──► FC(128)─ReLU ──► Price / Label
[GRU Text Features    (64-dim)] ──┘
```

**Files needed** (upload to Colab or mount Google Drive):
- `labeled_dataset.csv`
- `vit_image_features_meta.csv`
- `vit_image_features.npy`
- `gru_text_features.npy`

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# All libraries are pre-installed in Colab — no pip needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib, os, json, warnings
warnings.filterwarnings('ignore')

from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

print('✅ Libraries loaded')

## 📂 Step 2 — Upload Dataset Files

In [ ]:
# ── Option A: Upload manually ──────────────────────────────────────────────
from google.colab import files
print('Upload your 4 files: labeled_dataset.csv, vit_image_features_meta.csv,')
print('                      vit_image_features.npy, gru_text_features.npy')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── Option B: Mount Google Drive (comment out Option A above, use this instead)
# from google.drive import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/your_folder/'   # <-- change path

DATA_DIR = '.'   # current directory (works for Option A)

## 🔍 Step 3 — Load & Explore Data

In [ ]:
# ── Load CSVs ──────────────────────────────────────────────────────────────
labeled   = pd.read_csv(os.path.join(DATA_DIR, 'labeled_dataset.csv'))
meta      = pd.read_csv(os.path.join(DATA_DIR, 'vit_image_features_meta.csv'))

# ── Load pre-extracted feature arrays ─────────────────────────────────────
img_feats = np.load(os.path.join(DATA_DIR, 'vit_image_features.npy'))   # (5216, 768)
txt_feats = np.load(os.path.join(DATA_DIR, 'gru_text_features.npy'))    # (5358,  64)

print('=== labeled_dataset.csv ===')
print(f'  Shape  : {labeled.shape}')
print(f'  Columns: {list(labeled.columns)}')
display(labeled.head(3))

print('\n=== vit_image_features_meta.csv ===')
print(f'  Shape  : {meta.shape}')
print(f'  Columns: {list(meta.columns)}')
display(meta.head(3))

print(f'\nImage feature array : {img_feats.shape}  dtype={img_feats.dtype}')
print(f'Text feature array  : {txt_feats.shape}  dtype={txt_feats.dtype}')

In [ ]:
# ── Data exploration plots ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Price distribution
axes[0].hist(labeled['price_egp'], bins=60, color='#4A90D9', edgecolor='none')
axes[0].set_title('Price Distribution (EGP)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (EGP)'); axes[0].set_ylabel('Count')
axes[0].axvline(labeled['price_egp'].median(), color='red', linestyle='--', label=f'Median: {labeled["price_egp"].median():,.0f}')
axes[0].legend()

# Label counts
vc = labeled['label'].value_counts()
bars = axes[1].bar(vc.index, vc.values, color=['#4CAF50','#F44336'], edgecolor='none')
for bar, val in zip(bars, vc.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 f'{val}', ha='center', fontweight='bold')
axes[1].set_title('Label Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')

# Brand breakdown
top_brands = labeled['brand'].value_counts().head(8)
axes[2].barh(top_brands.index[::-1], top_brands.values[::-1], color='#9B59B6')
axes[2].set_title('Top Brands', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nPrice stats:')
print(labeled['price_egp'].describe().round(0))

## 🔗 Step 4 — Feature Alignment & Multimodal Fusion

In [ ]:
# ── Build listing_id → text feature row index ──────────────────────────────
# gru_text_features.npy rows match labeled_dataset.csv row order
lid_to_txt = {row.listing_id: i for i, row in labeled.reset_index(drop=True).iterrows()}

# ── Merge meta with txt index ──────────────────────────────────────────────
df = meta[['listing_id', 'feat_index', 'price_egp', 'label_int']].copy()
df['txt_idx'] = df['listing_id'].map(lid_to_txt)
df = df.dropna(subset=['txt_idx'])
df['txt_idx'] = df['txt_idx'].astype(int)

print(f'Aligned samples  : {len(df):,}')
print(f'Dropped (no match): {len(meta)-len(df)}')

# ── Extract aligned feature arrays ────────────────────────────────────────
img_X = img_feats[df['feat_index'].values]   # (N, 768) — ViT
txt_X = txt_feats[df['txt_idx'].values]      # (N,  64) — GRU

# ── EARLY FUSION: Concatenate ──────────────────────────────────────────────
#
#   [ViT 768] + [GRU 64] = [Fused 832]
#
X = np.concatenate([img_X, txt_X], axis=1)  # (N, 832)
y_price = df['price_egp'].values             # regression target
y_label = df['label_int'].values             # classification target (1=Fair, 2=Overpriced)

print(f'\nFused feature matrix: {X.shape}')
print(f'  → Image features  : {img_X.shape}')
print(f'  → Text features   : {txt_X.shape}')
print(f'Price range         : {y_price.min():,.0f} – {y_price.max():,.0f} EGP')
print(f'Labels              : { {1:"Fair", 2:"Overpriced"} }')

In [ ]:
# ── Architecture diagram ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.axis('off')

BOXES = [
    # (x, y, w, h, label, color)
    (0.01, 0.62, 0.13, 0.22, 'ViT Image\nFeatures\n(768-dim)', '#AED6F1'),
    (0.01, 0.16, 0.13, 0.22, 'GRU Text\nFeatures\n(64-dim)',  '#A9DFBF'),
    (0.20, 0.38, 0.10, 0.24, 'Concat\n(832)',                  '#F9E79F'),
    (0.36, 0.38, 0.10, 0.24, 'FC Layer\n512\nReLU',            '#F0B27A'),
    (0.50, 0.38, 0.10, 0.24, 'FC Layer\n256\nReLU',            '#F0B27A'),
    (0.64, 0.38, 0.10, 0.24, 'FC Layer\n128\nReLU',            '#F0B27A'),
    (0.80, 0.62, 0.17, 0.18, '💰 Price\nPrediction (EGP)',     '#D2B4DE'),
    (0.80, 0.20, 0.17, 0.18, '🏷️ Label\nFair / Overpriced',   '#FADBD8'),
]
for (x,y,w,h,txt,col) in BOXES:
    rect = mpatches.FancyBboxPatch((x,y),w,h,
        boxstyle='round,pad=0.015',linewidth=1.2,edgecolor='#555',facecolor=col)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, txt, ha='center', va='center', fontsize=8.5, fontweight='bold')

ARROWS = [
    (0.14,0.73, 0.20,0.55), (0.14,0.27, 0.20,0.45),
    (0.30,0.50, 0.36,0.50), (0.46,0.50, 0.50,0.50),
    (0.60,0.50, 0.64,0.50), (0.74,0.55, 0.80,0.71),
    (0.74,0.45, 0.80,0.29),
]
for (x1,y1,x2,y2) in ARROWS:
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
        arrowprops=dict(arrowstyle='->', color='#333',lw=1.8))

ax.set_title('Multimodal Fusion Architecture — Early Fusion (Concatenation)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## ✂️ Step 5 — Train / Test Split & Scaling

In [ ]:
# ── Split ──────────────────────────────────────────────────────────────────
X_train, X_test, yp_train, yp_test, yl_train, yl_test = train_test_split(
    X, y_price, y_label, test_size=0.2, random_state=42, stratify=y_label)

# ── Scale ──────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit only on train!
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test : {X_test.shape[0]:,} samples')
print(f'Feature dim: {X_train.shape[1]}')

## 🧠 Step 6 — Build & Train the Multimodal MLP

In [ ]:
# ══════════════════════════════════════════════════════
#  HEAD 1 — Price Regression
#  Input: fused (832)  →  FC(512)→FC(256)→FC(128)→1
# ══════════════════════════════════════════════════════
print('🔧 Training Price Regressor ...')
price_model = MLPRegressor(
    hidden_layer_sizes = (512, 256, 128),
    activation         = 'relu',
    solver             = 'adam',
    learning_rate_init = 0.001,
    max_iter           = 500,
    early_stopping     = True,
    validation_fraction= 0.1,
    n_iter_no_change   = 20,
    random_state       = 42,
    verbose            = False,
)
price_model.fit(X_train_s, yp_train)
print(f'  ✅ Converged in {price_model.n_iter_} iterations')

# ══════════════════════════════════════════════════════
#  HEAD 2 — Label Classifier
#  Input: fused (832)  →  FC(512)→FC(256)→FC(128)→2
# ══════════════════════════════════════════════════════
print('\n🔧 Training Label Classifier ...')
label_model = MLPClassifier(
    hidden_layer_sizes = (512, 256, 128),
    activation         = 'relu',
    solver             = 'adam',
    learning_rate_init = 0.001,
    max_iter           = 500,
    early_stopping     = True,
    validation_fraction= 0.1,
    n_iter_no_change   = 20,
    random_state       = 42,
    verbose            = False,
)
label_model.fit(X_train_s, yl_train)
print(f'  ✅ Converged in {label_model.n_iter_} iterations')

## 📊 Step 7 — Evaluate Results

In [ ]:
LABEL_MAP = {1: 'Fair', 2: 'Overpriced'}

# ── Price metrics ──────────────────────────────────────────────────────────
pred_prices = price_model.predict(X_test_s)
mae  = mean_absolute_error(yp_test, pred_prices)
rmse = np.sqrt(mean_squared_error(yp_test, pred_prices))
r2   = r2_score(yp_test, pred_prices)

# ── Label metrics ──────────────────────────────────────────────────────────
pred_labels = label_model.predict(X_test_s)
acc = accuracy_score(yl_test, pred_labels)

print('='*50)
print('  PRICE REGRESSION RESULTS')
print('='*50)
print(f'  MAE  (Mean Abs Error) : {mae:>12,.0f} EGP')
print(f'  RMSE                  : {rmse:>12,.0f} EGP')
print(f'  R²   (Variance expl.) : {r2:>12.4f}')
print()
print('='*50)
print('  LABEL CLASSIFICATION RESULTS')
print('='*50)
print(f'  Accuracy              : {acc:>12.2%}')
print()
print(classification_report(yl_test, pred_labels,
      target_names=[LABEL_MAP[1], LABEL_MAP[2]]))

In [ ]:
# ── Evaluation Plots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Actual vs Predicted price
colors = ['#4CAF50' if l == 1 else '#F44336' for l in pred_labels]
axes[0].scatter(yp_test, pred_prices, c=colors, alpha=0.4, s=18, edgecolors='none')
lim = max(yp_test.max(), pred_prices.max()) * 1.05
axes[0].plot([0,lim],[0,lim], 'k--', lw=1.5, label='Perfect')
axes[0].set_xlim(0,lim); axes[0].set_ylim(0,lim)
axes[0].set_xlabel('Actual Price (EGP)', fontsize=11)
axes[0].set_ylabel('Predicted Price (EGP)', fontsize=11)
axes[0].set_title(f'Actual vs Predicted Price\nR² = {r2:.4f}', fontsize=12, fontweight='bold')
fair_p = mpatches.Patch(color='#4CAF50', label='Predicted: Fair')
over_p = mpatches.Patch(color='#F44336', label='Predicted: Overpriced')
axes[0].legend(handles=[fair_p, over_p], fontsize=9)

# 2. Prediction error distribution
errors = pred_prices - yp_test
axes[1].hist(errors, bins=50, color='#9B59B6', edgecolor='none', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', lw=1.5)
axes[1].axvline(mae, color='red', linestyle='--', lw=1.2, label=f'MAE = {mae:,.0f}')
axes[1].axvline(-mae, color='red', linestyle='--', lw=1.2)
axes[1].set_title('Prediction Error Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Error (EGP)'); axes[1].set_ylabel('Count')
axes[1].legend(fontsize=9)

# 3. Confusion matrix
cm = confusion_matrix(yl_test, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
       display_labels=[LABEL_MAP[1], LABEL_MAP[2]])
disp.plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title(f'Label Confusion Matrix\nAccuracy = {acc:.2%}',
                  fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Step 8 — Save Model Files

In [ ]:
os.makedirs('model', exist_ok=True)

joblib.dump(scaler,      'model/scaler.pkl')
joblib.dump(price_model, 'model/price_regressor.pkl')
joblib.dump(label_model, 'model/label_classifier.pkl')

meta_info = {
    'img_feat_dim': 768,
    'txt_feat_dim': 64,
    'total_dim':    832,
    'fusion':       'early_concatenation',
    'hidden_layers': [512, 256, 128],
    'label_map':    {'1': 'Fair', '2': 'Overpriced'},
    'mae':          round(float(mae), 2),
    'rmse':         round(float(rmse), 2),
    'r2':           round(float(r2), 4),
    'accuracy':     round(float(acc), 4),
    'n_train':      int(X_train.shape[0]),
    'n_test':       int(X_test.shape[0]),
}
with open('model/model_meta.json', 'w') as f:
    json.dump(meta_info, f, indent=2)

print('✅ Saved:')
for fn in ['scaler.pkl', 'price_regressor.pkl', 'label_classifier.pkl', 'model_meta.json']:
    size = os.path.getsize(f'model/{fn}')
    print(f'   model/{fn}  ({size/1024:.1f} KB)')

# ── Download all model files from Colab ────────────────────────────────────
import zipfile
with zipfile.ZipFile('multimodal_model.zip', 'w') as zf:
    for fn in os.listdir('model'):
        zf.write(f'model/{fn}')
print('\n📦 multimodal_model.zip ready for download')
files.download('multimodal_model.zip')

## 🔮 Step 9 — Single Listing Inference

In [ ]:
LABEL_MAP = {1: 'Fair', 2: 'Overpriced'}

def predict_listing(img_vector, txt_vector):
    """
    Given a ViT image feature vector (768,) and a GRU text feature vector (64,),
    return predicted price and label.
    """
    # 1. Fuse features
    fused = np.concatenate([img_vector, txt_vector]).reshape(1, -1)
    # 2. Scale
    fused_s = scaler.transform(fused)
    # 3. Predict
    price  = float(price_model.predict(fused_s)[0])
    label  = int(label_model.predict(fused_s)[0])
    proba  = float(label_model.predict_proba(fused_s)[0].max())
    return {
        'predicted_price_egp': round(price, 2),
        'predicted_label':     LABEL_MAP[label],
        'confidence':          round(proba, 4),
    }


# ── Pick a random test listing and run inference ───────────────────────────
idx         = np.random.randint(len(X_test))
sample_img  = X_test[idx, :768]    # ViT slice
sample_txt  = X_test[idx, 768:]    # GRU slice
actual_price = yp_test[idx]
actual_label = LABEL_MAP[int(yl_test[idx])]

result = predict_listing(sample_img, sample_txt)

print('━'*40)
print('       INFERENCE EXAMPLE')
print('━'*40)
print(f'  Actual  price : {actual_price:>10,.0f} EGP')
print(f'  Predicted price: {result["predicted_price_egp"]:>9,.0f} EGP')
print(f'  Error          : {abs(result["predicted_price_egp"]-actual_price):>9,.0f} EGP')
print()
print(f'  Actual  label : {actual_label}')
print(f'  Predicted label: {result["predicted_label"]}')
print(f'  Confidence     : {result["confidence"]:.2%}')
print('━'*40)

## 📊 Step 10 — Batch Inference on Test Set

In [ ]:
# Run full test set and build result table
all_prices = price_model.predict(X_test_s)
all_labels = label_model.predict(X_test_s)
all_probs  = label_model.predict_proba(X_test_s).max(axis=1)

results_df = pd.DataFrame({
    'actual_price_egp':    yp_test,
    'predicted_price_egp': np.round(all_prices, 0),
    'actual_label':        [LABEL_MAP[l] for l in yl_test],
    'predicted_label':     [LABEL_MAP[l] for l in all_labels],
    'confidence':          np.round(all_probs, 3),
})
results_df['price_error_egp'] = (results_df['predicted_price_egp'] - results_df['actual_price_egp']).round(0)
results_df['correct']         = results_df['actual_label'] == results_df['predicted_label']

print(f'Test samples  : {len(results_df):,}')
print(f'Label accuracy: {results_df["correct"].mean():.2%}')
print(f'Price MAE     : {results_df["price_error_egp"].abs().mean():,.0f} EGP')
display(results_df.head(10))